# Lab 4: Imaging AI

**AI2Practice Research Lab** · about 2 hours · runs in Google Colab

## The clinical question

**Can a model tell pneumonia from normal on a chest X-ray, and what is it looking at?**

Pneumonia is an infection of the lung. On a chest X-ray it often shows up as a white, cloudy area (an *opacity*) where the lung should look dark. In this lab you will train a small image model to sort chest X-rays into "normal" and "pneumonia", measure how well it does, look at which pixels it relies on, and then test it on images that look as if they came from a different hospital.

## What you will be able to do

By the end you will be able to explain how a picture becomes a grid of numbers, explain why a training/test split must be done by patient and not by image, train and evaluate a small convolutional neural network, draw a saliency map, and show how a model's accuracy can fall when the images change.

## How this notebook works

Every task starts with a **Mode** line telling you what kind of task it is. **Write** means you write the code yourself from an empty cell. **Fill in** means you finish code that is partly written (replace each `___`). **Reuse** means you use a function you wrote in an earlier lab, saved in your `toolkit.py`. **Direct the AI** means you ask an AI assistant to write the code, paste it in, run the test, and fill in a short log. **Fix** means you find and repair a bug in code "written by an AI". **Read** means you predict what code will do before you run it.

Each task ends with a **test cell**. Run it: if it prints `Check passed`, move on. If it shows an error, read the message, fix your code, and try again. Hints and solutions are hidden in collapsed sections; try first, then peek.

To run a cell, click it and press **Shift + Enter**. Run the cells in order from top to bottom.

**Tip:** this lab trains a neural network. It works on Colab's normal CPU, but it is faster on a GPU: *Runtime > Change runtime type > T4 GPU*. Do this before you start, because changing it restarts the notebook.

## Set your GitHub username

Type your GitHub username between the quotes and run the cell. It is used to find the `toolkit.py` you saved in your fork in earlier labs.

In [ ]:
GITHUB_USERNAME = "your-username"

## Setup

This cell installs **MedMNIST** (a free collection of small, ready-to-use medical images), downloads your `toolkit.py`, and loads the libraries we need. **PyTorch** is the library we use to build and train the neural network; it is already installed on Colab. It may take a minute the first time.

In [ ]:
import os, sys, subprocess, urllib.request

# 1. Install MedMNIST if it is not already installed
try:
    import medmnist
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "medmnist"])
    import medmnist

# 2. Get toolkit.py: first from your own fork, then from the course repo
if not os.path.exists("toolkit.py"):
    for url in [
        f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/CA2i-Research-Launchpad/main/lesson-notebooks/toolkit.py",
        "https://raw.githubusercontent.com/UM-IHC-CA2i/CA2i-Research-Launchpad/main/lesson-notebooks/toolkit.py",
    ]:
        try:
            urllib.request.urlretrieve(url, "toolkit.py")
            print("Downloaded toolkit.py from", url)
            break
        except Exception:
            pass

# 3. Import sens_spec from your toolkit (written in Lab 3).
#    If that fails, use a simple stand-in so the notebook still runs.
try:
    from toolkit import sens_spec
    print("Using sens_spec from your toolkit.py")
except Exception:
    def sens_spec(y_true, y_pred):
        """Return (sensitivity, specificity) for 0/1 labels."""
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
        tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
        sens = tp / (tp + fn) if (tp + fn) else float("nan")
        spec = tn / (tn + fp) if (tn + fp) else float("nan")
        return sens, spec
    print("toolkit.py not found; using a stand-in sens_spec")

# 4. Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("MedMNIST version:", medmnist.__version__)
print("Training will run on:", device)

## Load the data

**PneumoniaMNIST** is a set of 5,856 pediatric chest X-rays, shrunk to 28 by 28 pixels so they are quick to work with. Each image has a **label**, the correct answer we want the model to learn: `0` means normal and `1` means pneumonia. The creators of the dataset have already divided the images into three groups, called **splits**: a **training set** the model learns from, a **validation set** we use to check progress while training, and a **test set** we keep aside until the very end to measure how well the model works on images it has never seen.

In [ ]:
from medmnist import PneumoniaMNIST, INFO

train_ds = PneumoniaMNIST(split="train", download=True)
val_ds   = PneumoniaMNIST(split="val",   download=True)
test_ds  = PneumoniaMNIST(split="test",  download=True)

# The images and labels as NumPy arrays (NumPy is Python's library for grids of numbers)
train_imgs, train_labels = train_ds.imgs, train_ds.labels.ravel()
val_imgs,   val_labels   = val_ds.imgs,   val_ds.labels.ravel()
test_imgs,  test_labels  = test_ds.imgs,  test_ds.labels.ravel()

print("Label meanings:", INFO["pneumoniamnist"]["label"])
print("Training images:  ", len(train_imgs))
print("Validation images:", len(val_imgs))
print("Test images:      ", len(test_imgs))

---
# Part 1: Images as data

To a computer, a grayscale image is just a table of numbers. Each small square in the picture is a **pixel**, and each pixel holds one number, its **intensity**: how bright it is. In these images the numbers run from 0 (black) to 255 (white). On a chest X-ray, air in the lungs lets X-rays through and looks dark (low numbers), while bone and fluid block them and look bright (high numbers). A pneumonia opacity is a patch of higher numbers where the lung should be dark.

Let us look at one image, first as a picture and then as the numbers behind it.

In [ ]:
img = train_imgs[0]
name = INFO["pneumoniamnist"]["label"][str(train_labels[0])]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Training image 0 (label: {name})")
axes[0].axis("off")
axes[1].hist(img.ravel(), bins=32, color="gray")
axes[1].set_title("How many pixels have each intensity")
axes[1].set_xlabel("Pixel intensity (0 = black, 255 = white)")
plt.show()

# The top-left 6 x 6 corner of the image, as numbers
print(img[:6, :6])

### Task 1.1: Predict the output

**Mode: Read**

Before you run the next cell, write down your predictions. What will `img.shape` print? (`shape` gives the number of rows and columns in a grid of numbers.) What are the smallest and largest values you could possibly see? Then run it and compare.

In [ ]:
print("Shape:", img.shape)
print("Smallest value:", img.min(), " Largest value:", img.max())
print("Data type:", img.dtype)

<details><summary>Explanation</summary>

The shape is `(28, 28)`: 28 rows and 28 columns, so 784 pixels in total. The data type `uint8` means "unsigned 8-bit integer", whole numbers from 0 to 255. A real chest X-ray is often 2,000 by 2,500 pixels or more with a wider range of values; these are shrunk so they train quickly. The cost is that fine detail is lost, which matters when you judge the results.
</details>

In [ ]:
assert img.shape == (28, 28)
assert img.dtype == np.uint8
print("Check passed")

### Task 1.2: How many training images show pneumonia?

**Mode: Write**

Before training any model, always look at the balance of labels. Create a variable `n_pneumonia_train` holding the number of training images labeled pneumonia (label `1`), and print it along with the fraction of all training images it represents.

<details><summary>Hint</summary>

`train_labels` is an array of 0s and 1s. `train_labels == 1` gives True/False for each image, and `.sum()` counts the Trues. Because the labels are 0 and 1, `train_labels.sum()` also works.
</details>

<details><summary>Solution</summary>

```python
n_pneumonia_train = int((train_labels == 1).sum())
print(n_pneumonia_train, n_pneumonia_train / len(train_labels))
```
</details>

In [ ]:
# Your code here

In [ ]:
assert n_pneumonia_train == int(train_labels.sum()), "Count the images whose label is 1."
print("Check passed")
print(f"About {100 * n_pneumonia_train / len(train_labels):.0f}% of training images show pneumonia.")

About three in four training images show pneumonia. That matters: a lazy "model" that always answers "pneumonia" would already be right about 74% of the time on the training images. Any real model has to beat that.

### Task 1.3: Window and level

**Mode: Fill in**

When you read a CT scan, you adjust the **window** and **level** to choose which range of values fills the screen from black to white (Lesson 2.2, *Images as Data*, on the AI2Practice site, has an interactive version). The **level** is the value at the middle of the range, and the **width** is how wide the range is. Everything below the range is shown black, everything above it white, and values inside are stretched across the full gray scale. A narrow window gives high contrast for a small range of tissues.

CT values are in Hounsfield units, while these X-ray pixels are 0 to 255, but the idea is the same. Fill in the three blanks (`___`) in the function below.

<details><summary>Hint</summary>

The bottom of the range is `level - width / 2` and the top is `level + width / 2`. `np.clip(x, low, high)` replaces anything below `low` with `low` and anything above `high` with `high`.
</details>

<details><summary>Solution</summary>

```python
low  = level - width / 2
high = level + width / 2
out = np.clip(image.astype(float), low, high)
```
</details>

In [ ]:
def apply_window(image, level, width):
    """Show only pixel values from level - width/2 to level + width/2, stretched to 0-255."""
    low  = ___
    high = ___
    out = ___                               # clip the image to [low, high]
    out = (out - low) / (high - low) * 255  # stretch to 0-255
    return out.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (level, width) in zip(axes, [(128, 256), (100, 80), (180, 60)]):
    ax.imshow(apply_window(img, level, width), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"level {level}, width {width}")
    ax.axis("off")
plt.show()

In [ ]:
test_img = np.array([[0, 50, 100], [150, 200, 255]], dtype=np.uint8)
w = apply_window(test_img, level=100, width=100)   # range 50 to 150
assert w[0, 0] == 0 and w[0, 1] == 0, "Values at or below the bottom of the range should be black (0)."
assert w[1, 1] == 255 and w[1, 2] == 255, "Values at or above the top of the range should be white (255)."
assert w[0, 2] == 127, "The level (100) should land in the middle of the gray scale."
print("Check passed")

**Optional:** run the next cell to get sliders and try your own window and level. (It uses `ipywidgets`, which Colab includes.)

In [ ]:
try:
    from ipywidgets import interact
    def show(level=128, width=256):
        plt.imshow(apply_window(img, level, width), cmap="gray", vmin=0, vmax=255)
        plt.axis("off"); plt.show()
    interact(show, level=(0, 255, 5), width=(10, 400, 10))
except ImportError:
    print("ipywidgets is not available here; skip this optional cell.")

---
# Part 2: Splitting by patient

A model is only useful if it works on **new patients**. So we test it on images it never saw during training. But there is a trap. Many patients have more than one X-ray: a follow-up film, a repeat because the first was rotated, a lateral view. Two X-rays of the same person look much more alike than X-rays of two different people, down to the shape of the ribs, a pacemaker, or the way the technologist positioned them.

If we split the images at random, one image of a patient can land in the training set and another image of the same patient in the test set. The model can then score well by recognizing the *patient* rather than the *disease*. This is called **data leakage**: information from the test set leaking into training. It makes results look better than they really are, and it is one of the most common errors in published medical imaging AI papers. The fix is to split by **patient**: every image of a given patient goes into the same set.

The PneumoniaMNIST files do not include patient numbers. Instead, the dataset's creators fixed the splits in advance (the test set comes from the original dataset's separate test folder), and everyone uses the same **official splits**. That makes results comparable between studies, and it is why we use them as they are and never re-shuffle them. To practice the patient-level split, we will use a small made-up table that does have patient numbers.

In [ ]:
# A made-up list of 1,000 X-ray images from a few hundred patients (many patients have several images)
rng = np.random.default_rng(4)
patient_of_image = rng.integers(1, 401, size=1000)
images = pd.DataFrame({
    "image_id":   [f"IMG{i:04d}" for i in range(1000)],
    "patient_id": [f"P{p:03d}" for p in patient_of_image],
})
print(images.head())
print("Images:", len(images), " Patients:", images.patient_id.nunique())

### Task 2.1: Fix the AI's split

**Mode: Fix**

A student asked an AI assistant: *"Split my images into 80% training and 20% test."* The AI wrote the code below. It runs without an error, but it has a serious problem. Run it, read the printed check, then fix the code so that **no patient appears in both sets**, with roughly 20% of the *patients* in the test set.

<details><summary>Hint</summary>

Split the list of unique patient IDs (`images.patient_id.unique()`) instead of the rows. Then keep the rows whose `patient_id` is in the test patients for the test set, and the rest for training. `images.patient_id.isin(some_list)` gives True for rows whose patient is in the list.
</details>

<details><summary>Solution</summary>

```python
from sklearn.model_selection import train_test_split

patients = images.patient_id.unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=0)
train_df = images[images.patient_id.isin(train_patients)]
test_df  = images[images.patient_id.isin(test_patients)]
```

scikit-learn also has a ready-made tool for this, `GroupShuffleSplit`, which splits rows while keeping each group (here, each patient) together.
</details>

In [ ]:
from sklearn.model_selection import train_test_split

# Code written by an AI assistant
train_df, test_df = train_test_split(images, test_size=0.2, random_state=0)

shared = set(train_df.patient_id) & set(test_df.patient_id)
print("Training images:", len(train_df), " Test images:", len(test_df))
print("Patients who appear in BOTH sets:", len(shared))

In [ ]:
assert len(train_df) + len(test_df) == len(images), "Every image should end up in exactly one set."
assert set(train_df.patient_id).isdisjoint(test_df.patient_id), "Some patients are still in both training and test."
frac = test_df.patient_id.nunique() / images.patient_id.nunique()
assert 0.15 <= frac <= 0.25, f"About 20% of patients should be in the test set (you have {frac:.0%})."
print("Check passed")

### Task 2.2: Count the labels in each official split

**Mode: Direct the AI**

Ask an AI assistant to write the code for you. In Colab, click the **Gemini** button or the "generate" link in an empty code cell; any other assistant is fine too. You need to tell it what your variables are called. A good prompt might start like this:

*"I have NumPy arrays `train_labels`, `val_labels` and `test_labels` containing 0 (normal) and 1 (pneumonia). Write Python code that makes a pandas DataFrame called `counts` with rows 'train', 'val', 'test' and columns 'normal' and 'pneumonia', holding the number of images of each kind..."*

Paste the AI's code in the empty cell, run it, and then run the test cell. If the test fails, tell the AI what went wrong, or fix the code yourself. Then fill in the AI-use log.

<details><summary>One possible solution</summary>

```python
counts = pd.DataFrame({
    "normal":    [int((l == 0).sum()) for l in [train_labels, val_labels, test_labels]],
    "pneumonia": [int((l == 1).sum()) for l in [train_labels, val_labels, test_labels]],
}, index=["train", "val", "test"])
print(counts)
```
</details>

In [ ]:
# Paste the AI's code here

In [ ]:
assert list(counts.index) == ["train", "val", "test"], "Rows should be 'train', 'val', 'test' in that order."
assert list(counts.columns) == ["normal", "pneumonia"], "Columns should be 'normal' and 'pneumonia'."
assert counts.values.sum() == len(train_labels) + len(val_labels) + len(test_labels)
assert counts.loc["test"].sum() == len(test_labels)
assert counts.loc["test", "pneumonia"] == int(test_labels.sum())
print("Check passed")
print(f"Test set: {counts.loc['test'].sum()} images, {counts.loc['test', 'pneumonia']} with pneumonia.")

**AI-use log, Task 2.2.** Double-click this cell and fill it in. Keeping a log like this is good research practice: anyone reading your work can see which code came from an AI and how you checked it.

*Prompt I used:* (paste your prompt here)

*What I changed in the AI's code, and why:* (write "nothing" if you changed nothing)

*Did the test cell pass on the first try? If not, what went wrong?*

Notice that the test set (about 63% pneumonia) is balanced differently from the training set (about 74%). Keep the test numbers in mind; you will report them at the end.

---
# Part 3: A first image model

## What is a CNN?

A **neural network** is a model made of many simple calculations stacked in layers. Each calculation multiplies its inputs by numbers called **weights** and adds them up. Training means adjusting those weights, a little at a time, until the network's answers match the labels.

A **convolutional neural network (CNN)** is the kind of neural network most used for images. Its first layers slide small 3-by-3 "stencils", called **filters**, across the image. Each filter responds to a simple pattern, such as an edge or a bright spot. Later layers combine these into larger patterns, such as a rib border or a patch of opacity. At the end, a final layer turns everything into two scores, one for "normal" and one for "pneumonia", and the larger score wins.

Three more words you will need. The **loss** is a single number measuring how wrong the model's answers are on a batch of images; training tries to make it smaller. A **batch** is a small group of images (here 64) the model looks at before each weight update. An **epoch** is one complete pass through all the training images.

First we turn the images into PyTorch **tensors** (PyTorch's version of a NumPy array), scaled from 0-255 down to 0-1, with an extra dimension for the single gray "color channel".

In [ ]:
def to_tensor(imgs):
    return torch.tensor(imgs, dtype=torch.float32).unsqueeze(1) / 255.0   # shape: (N, 1, 28, 28)

X_train, y_train = to_tensor(train_imgs), torch.tensor(train_labels, dtype=torch.long)
X_val,   y_val   = to_tensor(val_imgs),   torch.tensor(val_labels,   dtype=torch.long)
X_test,  y_test  = to_tensor(test_imgs),  torch.tensor(test_labels,  dtype=torch.long)
print("X_train shape:", tuple(X_train.shape))

Here is the model. You do not need to understand every line. Read the comments to see the two filter layers, each followed by **pooling** (which shrinks the image by half, keeping the strongest response in each 2-by-2 block), and then the final layers that produce the two scores.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 16 filters look at the 28x28 image
            nn.ReLU(),                                    # keep positive responses only
            nn.MaxPool2d(2),                              # 28x28 -> 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 32 filters combine the first patterns
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                 # 32 x 7 x 7 grid -> one long list of numbers
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 2),                             # two scores: normal, pneumonia
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)
print(model)
print("Number of weights:", sum(p.numel() for p in model.parameters()))

### Task 3.1: The training loop

**Mode: Fill in**

Training repeats four steps for every batch: (1) the model makes predictions, (2) we compute the loss, (3) we work out in which direction each weight should move to reduce the loss (this is called **backpropagation**, and PyTorch does it with `loss.backward()`), and (4) the **optimizer** nudges the weights in that direction (`optimizer.step()`). Before step 3 we clear the directions left over from the previous batch with `optimizer.zero_grad()`.

Fill in the three blanks.

<details><summary>Hint</summary>

The loss function is called `loss_fn` and takes the model's scores and the true labels: `loss_fn(scores, yb)`. The other two blanks are the PyTorch calls named in the text above.
</details>

<details><summary>Solution</summary>

```python
            loss = loss_fn(scores, yb)
            optimizer.zero_grad()
            loss.backward()
```
</details>

In [ ]:
def accuracy(model, X, y):
    """Fraction of images the model labels correctly."""
    model.eval()
    with torch.no_grad():
        preds = model(X.to(device)).argmax(dim=1).cpu()
    return (preds == y).float().mean().item()

def train_model(model, X, y, epochs=5, batch_size=64, lr=1e-3, X_check=None, y_check=None):
    loss_fn = nn.CrossEntropyLoss()                        # the standard loss for picking between classes
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X))                     # shuffle the images every epoch
        total_loss = 0.0
        for start in range(0, len(X), batch_size):
            idx = order[start:start + batch_size]
            xb, yb = X[idx].to(device), y[idx].to(device)
            scores = model(xb)                             # step 1: predict
            loss = ___                               # step 2: how wrong?
            ___                                            # clear old directions
            ___                                            # step 3: backpropagation
            optimizer.step()                               # step 4: nudge the weights
            total_loss += loss.item() * len(idx)
        msg = f"epoch {epoch + 1}: average loss {total_loss / len(X):.3f}"
        if X_check is not None:
            msg += f", validation accuracy {accuracy(model, X_check, y_check):.1%}"
        print(msg)
    return model

model = SmallCNN().to(device)
model = train_model(model, X_train, y_train, epochs=5, X_check=X_val, y_check=y_val)

In [ ]:
val_acc = accuracy(model, X_val, y_val)
assert val_acc > 0.80, f"Validation accuracy is only {val_acc:.1%}. Check your blanks and run the training cell again."
print("Check passed")

Watch the loss go down epoch by epoch: the model is getting less wrong on the training images. The validation accuracy tells you whether that learning carries over to images it is not trained on. Your numbers will differ slightly from your classmates', because training starts from random weights and shuffles the images randomly.

### Task 3.2: Predictions on the test set

**Mode: Direct the AI**

Now ask an AI assistant for a function `predict(model, X)` that returns the model's predicted label (0 or 1) for every image in `X` as a NumPy array. Tell the AI: the model is a PyTorch model that outputs two scores per image; `X` is a tensor of shape (N, 1, 28, 28); the model is on the device stored in the variable `device`; it should switch the model to evaluation mode, turn off gradient tracking, process the images in batches, and return a NumPy array of 0s and 1s.

Paste the code, run it, run the test, and fill in the log.

<details><summary>One possible solution</summary>

```python
def predict(model, X, batch_size=256):
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            scores = model(X[start:start + batch_size].to(device))
            preds.append(scores.argmax(dim=1).cpu())
    return torch.cat(preds).numpy()
```
</details>

In [ ]:
# Paste the AI's code here

In [ ]:
y_pred = predict(model, X_test)
assert isinstance(y_pred, np.ndarray), "predict() should return a NumPy array."
assert y_pred.shape == (len(X_test),), "There should be one prediction per test image."
assert set(np.unique(y_pred)) <= {0, 1}, "Predictions should be 0 or 1."
assert abs((y_pred == test_labels).mean() - accuracy(model, X_test, y_test)) < 1e-6
test_acc = (y_pred == test_labels).mean()
print("Check passed")
print(f"Test accuracy: {test_acc:.1%}")

**AI-use log, Task 3.2.** Double-click this cell and fill it in. Keeping a log like this is good research practice: anyone reading your work can see which code came from an AI and how you checked it.

*Prompt I used:* (paste your prompt here)

*What I changed in the AI's code, and why:* (write "nothing" if you changed nothing)

*Did the test cell pass on the first try? If not, what went wrong?*

### Task 3.3: Sensitivity and specificity

**Mode: Reuse**

Accuracy hides the kind of mistakes a model makes. In Lab 3 you wrote `sens_spec(y_true, y_pred)` and saved it in your `toolkit.py`. **Sensitivity** is the fraction of pneumonia images the model catches; **specificity** is the fraction of normal images it correctly calls normal. Use your function on the test set: store the results in `sens` and `spec`, and print them as percentages.

<details><summary>Solution</summary>

```python
sens, spec = sens_spec(test_labels, y_pred)
print(f"Sensitivity {sens:.1%}, specificity {spec:.1%}")
```
</details>

In [ ]:
# Your code here

In [ ]:
tp = ((test_labels == 1) & (y_pred == 1)).sum(); fn = ((test_labels == 1) & (y_pred == 0)).sum()
tn = ((test_labels == 0) & (y_pred == 0)).sum(); fp = ((test_labels == 0) & (y_pred == 1)).sum()
assert abs(sens - tp / (tp + fn)) < 1e-6, "sens should be TP / (TP + FN)."
assert abs(spec - tn / (tn + fp)) < 1e-6, "spec should be TN / (TN + FP)."
print("Check passed")

Most small models on this dataset have high sensitivity and lower specificity: they call many normal children's X-rays "pneumonia". Part of the reason is the imbalance you counted in Task 1.2. In practice, would you rather this tool miss pneumonia or over-call it? The answer depends on how it would be used, which is a clinical decision, not a coding one.

---
# Part 4: Saliency maps and a second hospital

## What is the model looking at?

A good accuracy number does not tell you *why* the model gives its answer. A **saliency map** is one simple way to look inside. For each pixel, it asks: if this pixel were a little brighter or darker, how much would the model's score change? Pixels with a big effect are drawn bright. PyTorch can compute this directly: we ask for the **gradient** of the score with respect to the input image, which is exactly that "how much would it change" number for every pixel.

Saliency maps are rough, and they can be noisy. But they can reveal **shortcut learning**: a model that gets the right answer for the wrong reason. Famous examples include chest X-ray models that learned to read the hospital's text marker in the corner of the film, or the word "PORTABLE" (portable films are taken on sicker patients), instead of looking at the lungs.

### Task 4.1: A saliency map

**Mode: Direct the AI**

Ask an AI assistant for a function `saliency(model, x)` that takes the PyTorch model and **one** image `x` of shape (1, 28, 28), and returns a 28-by-28 NumPy array of the absolute gradient of the model's top class score with respect to the input pixels. Tell it the model is on `device` and outputs two scores per image.

<details><summary>One possible solution</summary>

```python
def saliency(model, x):
    model.eval()
    x = x.unsqueeze(0).to(device).clone().requires_grad_(True)   # add a batch dimension
    scores = model(x)
    top = scores.argmax(dim=1).item()
    scores[0, top].backward()
    return x.grad.abs()[0, 0].cpu().numpy()
```
</details>

In [ ]:
# Paste the AI's code here

In [ ]:
s = saliency(model, X_test[0])
assert isinstance(s, np.ndarray) and s.shape == (28, 28), "saliency() should return a 28 x 28 NumPy array."
assert (s >= 0).all(), "Use the absolute value of the gradient."
assert s.max() > 0, "The map is all zeros; check that the gradient is computed with respect to the input image."
print("Check passed")

**AI-use log, Task 4.1.** Double-click this cell and fill it in. Keeping a log like this is good research practice: anyone reading your work can see which code came from an AI and how you checked it.

*Prompt I used:* (paste your prompt here)

*What I changed in the AI's code, and why:* (write "nothing" if you changed nothing)

*Did the test cell pass on the first try? If not, what went wrong?*

Now look at the maps for a few test images. Bright areas are the pixels that most affect the model's answer.

In [ ]:
names = INFO["pneumoniamnist"]["label"]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for col, i in enumerate([0, 1, 2, 300, 400, 500]):
    axes[0, col].imshow(test_imgs[i], cmap="gray")
    axes[0, col].set_title(f"true: {names[str(test_labels[i])]}\nmodel: {names[str(y_pred[i])]}", fontsize=9)
    axes[1, col].imshow(saliency(model, X_test[i]), cmap="hot")
    axes[1, col].set_title("saliency", fontsize=9)
    for r in (0, 1): axes[r, col].axis("off")
plt.tight_layout(); plt.show()

**Think about it** (double-click and write your answer): Does the model focus on the lungs, or also on areas outside them, such as the edges of the image or the shoulders? Would a radiologist look at the same places?

*Your answer:*

## A second hospital

A model trained at one hospital is often used at another, where the X-ray machines, settings and patients differ. This change is called **dataset shift** (or *domain shift*). We do not have a second hospital's images, so we will simulate one: the same test images, but brighter and with less contrast, as if taken on a different machine with different processing.

In [ ]:
def second_hospital(X):
    """Simulate a different X-ray machine: brighter, with lower contrast."""
    return (X * 0.6 + 0.35).clamp(0, 1)

X_test_b = second_hospital(X_test)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(X_test[0, 0], cmap="gray", vmin=0, vmax=1); axes[0].set_title("Hospital A")
axes[1].imshow(X_test_b[0, 0], cmap="gray", vmin=0, vmax=1); axes[1].set_title("Hospital B (simulated)")
for ax in axes: ax.axis("off")
plt.show()

### Task 4.2: Fix the AI's evaluation

**Mode: Fix**

An AI assistant wrote this code to measure accuracy at Hospital B. It runs, but the number it prints is wrong. Find the bug and fix it so `acc_b` is the true accuracy at Hospital B.

<details><summary>Hint</summary>

Accuracy is the number of correct predictions divided by the number of **all** images. Look at what the code divides by.
</details>

<details><summary>Solution</summary>

```python
acc_b = (y_pred_b == test_labels).sum() / len(test_labels)
```
or simply `(y_pred_b == test_labels).mean()`.
</details>

In [ ]:
# Code written by an AI assistant
y_pred_b = predict(model, X_test_b)
acc_b = (y_pred_b == test_labels).sum() / test_labels.sum()
print(f"Hospital A accuracy: {test_acc:.1%}")
print(f"Hospital B accuracy: {acc_b:.1%}")

In [ ]:
assert 0 <= acc_b <= 1, "An accuracy can never be above 100%."
assert abs(acc_b - (y_pred_b == test_labels).mean()) < 1e-9, "Divide by the number of all test images."
print("Check passed")

### Task 4.3: Which mistakes did Hospital B bring?

**Mode: Reuse**

Use `sens_spec` again, this time on the Hospital B predictions `y_pred_b`. Store the results in `sens_b` and `spec_b` and print them next to the Hospital A values (`sens`, `spec`).

<details><summary>Solution</summary>

```python
sens_b, spec_b = sens_spec(test_labels, y_pred_b)
print(f"Hospital A: sensitivity {sens:.1%}, specificity {spec:.1%}")
print(f"Hospital B: sensitivity {sens_b:.1%}, specificity {spec_b:.1%}")
```
</details>

In [ ]:
# Your code here

In [ ]:
assert abs(sens_b - ((test_labels == 1) & (y_pred_b == 1)).sum() / (test_labels == 1).sum()) < 1e-6
assert abs(spec_b - ((test_labels == 0) & (y_pred_b == 0)).sum() / (test_labels == 0).sum()) < 1e-6
print("Check passed")

The images show the same children with the same diagnoses; only the brightness and contrast changed, and a radiologist would read them the same way. Yet the model's answers change, often pushing it toward calling almost everything one class. This is why a model must be tested on data from each place it will be used (an *external validation*), and why Lab 3 asked whether a tool works equally well at both sites.

## A shortcut, on purpose

Finally, let us build a model that cheats, to see what shortcut learning looks like. Imagine Hospital A uses a small bright marker on films from its pneumonia ward. We stamp a white square in the top-left corner of every *pneumonia* training image, train a new model, and test it twice: on test images marked the same way, and on test images from a hospital that uses no marker.

### Task 4.4: Predict the result

**Mode: Read**

Before running the next cell, predict: which test will show the higher accuracy? Where will the saliency map be brightest? Write your predictions here (double-click):

*My predictions:*

In [ ]:
def add_marker(X, labels):
    """Stamp a white 4x4 square in the top-left corner of every pneumonia image."""
    X = X.clone()
    X[torch.as_tensor(labels) == 1, :, 1:5, 1:5] = 1.0
    return X

X_train_marked = add_marker(X_train, train_labels)
X_test_marked  = add_marker(X_test, test_labels)

torch.manual_seed(1)
shortcut_model = train_model(SmallCNN().to(device), X_train_marked, y_train, epochs=3)

acc_marked   = (predict(shortcut_model, X_test_marked) == test_labels).mean()
acc_unmarked = (predict(shortcut_model, X_test) == test_labels).mean()
print(f"Test images with the marker (same hospital): {acc_marked:.1%}")
print(f"Test images without the marker (new hospital): {acc_unmarked:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
i = int(np.where(test_labels == 1)[0][0])
axes[0].imshow(X_test_marked[i, 0], cmap="gray"); axes[0].set_title("marked pneumonia image")
axes[1].imshow(saliency(shortcut_model, X_test_marked[i]), cmap="hot"); axes[1].set_title("saliency")
for ax in axes: ax.axis("off")
plt.show()

In [ ]:
assert acc_marked > acc_unmarked
print("Check passed")

The shortcut model looks excellent when the marker is present, because the marker gives the answer away. Without the marker it does much worse, and its saliency map lights up the corner instead of the lungs. Real shortcuts are rarely this obvious: they can be a scanner type, a patient position, a chest tube that is only present after treatment, or the text burned into the image. Saliency maps and testing at a second hospital are two of the simplest ways to catch them.

**Think about it:** Name one real feature of a chest X-ray, unrelated to the lungs themselves, that might be more common in pneumonia films and that a model could learn as a shortcut.

*Your answer:*

---
# Report back

Run the cell below and enter the three numbers on the Lab 4 page of the AI2Practice site. Your accuracy will differ from other learners' because training is random; any honest result is accepted.

In [ ]:
print("Number of test images:", len(test_labels))
print("Number of pneumonia images in the test set:", int(test_labels.sum()))
print(f"My test accuracy: {100 * test_acc:.1f}%")

# Save to GitHub

Save your finished notebook to your fork of the course repository so you keep a record of your work (and so a mentor can see it).

1. In Colab, click **File > Save a copy in GitHub**.
2. If asked, allow Colab to access your GitHub account.
3. Under **Repository**, choose your fork: `your-username/CA2i-Research-Launchpad` (your copy of `https://github.com/UM-IHC-CA2i/CA2i-Research-Launchpad`). Branch: `main`.
4. Set the **file path** to `lesson-notebooks/Lab_4_Imaging_AI.ipynb`.
5. Write a short **commit message**, such as "Finish Lab 4", and keep "Include a link to Colaboratory" ticked.
6. Click **OK**. Open your fork on GitHub to check that the notebook is there.

If you added any useful functions in this lab (for example `predict` or `saliency`), you can copy them into your `toolkit.py` for Lab 6.